# ATLOP baseline — 약점 정성 테스트

> **최종 업데이트**: 2026-07-13: 개선 모델 검증 셀 추가 — 개선 1(GCN, `re_model_gcn.py`)·개선 2(GAT, `re_model_gat.py`) 체크포인트를 로드해 테스트 1(multi-hop)을 baseline과 나란히 재실행. baseline 파일은 무수정 유지, 학습은 `train_graph.py`.
>
> 2026-07-13: 테스트 3(다단계 multi-hop 추론 — A→B, B→C 명시 시 A→C 예측 여부, 허구 지명으로 암기 지식 차단) probe 셀 추가.
>
> 2026-07-12: ATLOP의 알려진 약점 2가지(상호참조 신호 부족, 장거리·복잡 구문 추론 실패)를 임의 문서 각 1건으로 확인하는 probe 셀 추가.

학습이 끝난 ATLOP baseline이 **어떤 유형의 문서에서 실패하는지** 정성적으로 확인한다. 확인하는 약점 세 가지:

1. **상호참조(coreference) 신호 부족** — ATLOP은 vertexSet에 명시된 멘션 위치만 logsumexp로 풀링한다. 근거 문장의 주어가 대명사(She/He/It)인데 그 대명사가 멘션으로 등록돼 있지 않으면, 모델이 attention만으로 "She = Marie Curie"를 연결해야 해서 관계를 놓치기 쉽다.
2. **장거리·복잡 구문 추론 실패** — head와 tail이 여러 문장 떨어져 있고 사이에 교란 엔티티가 있거나, 관계 서술이 길게 꼬인 내포절 안에 있으면 Localized Context Pooling이 관련 문맥을 제대로 잡지 못한다.
3. **다단계(multi-hop) 추론 부재** — A→B, B→C가 각각 명시되어도 이를 조합한 A→C는 단일 pair 분류만 하는 구조상 잡기 어렵다. ATLOP은 (head, tail) 쌍별로 독립 분류하므로 다른 쌍의 예측 결과를 참조하는 경로가 없다.

**사전 조건**: GPU/Colab에서 학습을 마친 체크포인트가 `results/atlop.pt`에 있어야 한다.

```bash
python -m Scripts.atlop.train_re --epochs 30 --distant_limit 20000 --distant_epochs 1 --run_name atlop --save_model
```

체크포인트 없이 실행하면 미학습 가중치라 결과가 무의미하다(경고 출력됨).

In [1]:
# 셋업: 학습된 ATLOP 로드 + probe 헬퍼
# (프로젝트 루트에서 노트북 실행. 학습 시 --emb_size/--block_size를 바꿨다면 아래도 맞출 것)
import sys
from pathlib import Path

import torch
from transformers import AutoConfig, AutoModel, AutoTokenizer

ROOT = Path.cwd()
assert (ROOT / "data").exists(), "프로젝트 루트에서 노트북을 실행하세요"
sys.path.insert(0, str(ROOT))

from data.docred_io import build_rel2id, load_rel_info, NUM_CLASSES
from Scripts.atlop.preprocess import build_features
from Scripts.atlop.re_model import DocREModel
from Scripts.atlop.train_re import make_collate_fn

MODEL_NAME = "bert-base-cased"
CKPT = ROOT / "results" / "atlop.pt"  # train_re.py --save_model 산출물

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=NUM_CLASSES)
encoder = AutoModel.from_pretrained(MODEL_NAME, config=config, attn_implementation="eager")
config.cls_token_id = tokenizer.cls_token_id
config.sep_token_id = tokenizer.sep_token_id
model = DocREModel(config, encoder, num_labels=NUM_CLASSES).to(device)

if CKPT.exists():
    model.load_state_dict(torch.load(CKPT, map_location=device))
    print(f"체크포인트 로드 완료: {CKPT}")
else:
    print("[경고] results/atlop.pt 없음 — 미학습 가중치라 결과 무의미. "
          "GPU/Colab에서 --save_model 로 학습 후 체크포인트를 가져올 것.")
model.eval()

rel2id = build_rel2id()
id2rel = {v: k for k, v in rel2id.items()}
rel_names = load_rel_info()  # P-code -> 사람이 읽는 이름
collate = make_collate_fn(tokenizer.pad_token_id)


def probe(doc, pairs_of_interest=None):
    """DocRED 형식 dict 하나를 모델에 통과시켜 예측 triple을 출력.
    pairs_of_interest: 주목할 (h_idx, t_idx) 목록 — 검출 여부를 별도 표시."""
    feats = build_features([doc], tokenizer, rel2id, show_progress=False)
    batch = collate(feats)
    with torch.no_grad():
        preds = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            entity_pos=batch["entity_pos"],
            hts=batch["hts"],
        )[0].cpu()
    names = [v[0]["name"] for v in doc["vertexSet"]]
    found = []
    for (h, t), row in zip(feats[0]["hts"], preds):
        for r in range(1, NUM_CLASSES):
            if row[r] == 1:
                found.append((h, t, id2rel[r]))
    if not found:
        print("예측된 관계 없음 (모두 Na)")
    for h, t, p in found:
        mark = "  <-- 관심 쌍" if pairs_of_interest and (h, t) in pairs_of_interest else ""
        print(f"({names[h]}) --{p}:{rel_names.get(p, '?')}--> ({names[t]}){mark}")
    if pairs_of_interest:
        for h, t in pairs_of_interest:
            if not any(fh == h and ft == t for fh, ft, _ in found):
                print(f"[미검출] ({names[h]}, {names[t]}) 쌍에서 관계를 찾지 못함")
    return found

체크포인트 로드 완료: c:\Users\dohy3\OneDrive\바탕 화면\멋사NLP5기\04_실전프로젝트1\project1\results\atlop.pt


## 테스트 1 — 다단계(multi-hop) 관계 추론

문서에 **A→B, B→C 관계만 명시**되어 있을 때, 둘을 조합해야 나오는 **A→C를 예측하는지** 확인한다. DocRED 정답 레이블의 상당 비율이 이런 추론형(여러 문장의 사실을 결합해야 하는) 관계라서, 이 능력이 없으면 벤치마크에서 구조적으로 recall을 잃는다.

- 명시 1 (문장 0): `Veldora Castle` --**P131**(located in ...)--> `Marniva`
- 명시 2 (문장 1): `Marniva` --**P1376**(capital of) / **P17**(country)--> `Tressonia`
- 추론 목표: `Veldora Castle` --**P17**(country)--> `Tressonia` — 문서 어디에도 직접 서술되지 않음

**가공(허구) 지명을 쓰는 이유**: Eiffel Tower–Paris–France 같은 실존 체인이면 모델이 문서를 읽지 않고도 사전학습 때 암기한 세상 지식으로 P17을 맞힐 수 있다. 허구 지명은 그 지름길을 차단해서 순수한 문서 내 추론 능력만 측정한다.

해석:
| 결과 | 판정 |
|---|---|
| (0,1)·(1,2) 검출 + (0,2) 검출 | multi-hop 추론 성공 — 이 예시에서는 약점 미발현 |
| (0,1)·(1,2) 검출 + (0,2) **미검출** | **약점 3 확인** — 명시 관계는 읽지만 조합(bridge) 추론은 못 함 |
| (0,1) 또는 (1,2)부터 미검출 | 판정 보류 — 허구 지명(OOV subword)이 원인일 수 있으니 지명을 바꿔 재시도 |

In [2]:
# 테스트 1: 다단계(multi-hop) 관계 추론 — A→B, B→C 명시, A→C는 추론해야만 나옴
# 문장 0: A(Veldora Castle)가 B(Marniva)에 있다 → P131
# 문장 1: B(Marniva)는 C(Tressonia)의 수도다 → P1376 / P17
# 어디에도 "Veldora Castle ... Tressonia"를 직접 잇는 서술은 없음.
# 허구 지명이라 사전학습 지식으로는 못 맞히고, 문서 내 조합 추론만이 유일한 경로.
doc_multihop = {
    "title": "probe_multihop",
    "sents": [
        ["Veldora", "Castle", "is", "located", "in", "Marniva", "."],
        ["Marniva", "is", "the", "capital", "of", "Tressonia", "."],
    ],
    "vertexSet": [
        [{"name": "Veldora Castle", "sent_id": 0, "pos": [0, 2], "type": "LOC"}],
        [{"name": "Marniva", "sent_id": 0, "pos": [5, 6], "type": "LOC"},
         {"name": "Marniva", "sent_id": 1, "pos": [0, 1], "type": "LOC"}],
        [{"name": "Tressonia", "sent_id": 1, "pos": [5, 6], "type": "LOC"}],
    ],
    "labels": [],
}
print("명시 1 기대: (Veldora Castle) --P131:located in ...--> (Marniva)")
print("명시 2 기대: (Marniva) --P1376:capital of / P17:country--> (Tressonia)")
print("추론 목표  : (Veldora Castle) --P17:country--> (Tressonia)  <- 문서에 직접 서술 없음")
print()
probe(doc_multihop, pairs_of_interest=[(0, 1), (1, 2), (0, 2)]);

명시 1 기대: (Veldora Castle) --P131:located in ...--> (Marniva)
명시 2 기대: (Marniva) --P1376:capital of / P17:country--> (Tressonia)
추론 목표  : (Veldora Castle) --P17:country--> (Tressonia)  <- 문서에 직접 서술 없음

(Tressonia) --P36:capital--> (Marniva)
[미검출] (Veldora Castle, Marniva) 쌍에서 관계를 찾지 못함
[미검출] (Marniva, Tressonia) 쌍에서 관계를 찾지 못함
[미검출] (Veldora Castle, Tressonia) 쌍에서 관계를 찾지 못함


## 개선 모델 검증 — 테스트 1 재실행 (Entity Pair Graph)

테스트 1에서 확인한 대로, ATLOP은 (h, t) 쌍을 서로 **독립으로** 분류하므로 A→B, B→C를 조합해 A→C를 만드는 경로가 구조적으로 없다. 두 개선 모델은 **baseline 파일(`re_model.py`/`train_re.py`)을 수정하지 않고**(상속) 쌍 표현 위에 **entity-pair graph**를 얹는다 — 엔티티를 공유하는 쌍끼리 연결(same-head / same-tail / bridge 3타입 엣지)해서, (A,C) 쌍 노드가 전제 (A,B)·(B,C) 노드의 정보를 한 layer 만에 읽을 수 있게 한다. 그래프 출력은 zero-init 잔차 헤드로 baseline logits에 더해지므로, 학습 시작점은 정확히 baseline이고 그래프가 도움이 되는 방향으로만 벗어난다.

| | 파일 / 클래스 | 구조 | 이웃 집계 방식 |
|---|---|---|---|
| 개선 1 | `Scripts/atlop/re_model_gcn.py` `DocREModelGCN` | GREP: Entity Pair Graph + relational GCN | 엣지 타입별 **고정 평균** |
| 개선 2 | `Scripts/atlop/re_model_gat.py` `DocREModelGAT` | Localized Context Pooling + Entity Pair Graph + GAT | **multi-head attention** (타입별 bias로 bridge 엣지 특화 가능) |

두 모델은 노드 특징·그래프 구조가 동일하고 집계 방식만 다르다 — GCN vs GAT의 깨끗한 ablation.

**학습** (Colab A100, baseline과 동일 레시피 — `train_graph.py`는 별도 진입점):

```bash
python -m Scripts.atlop.train_graph --model gcn --epochs 15 --distant_limit 20000 --distant_epochs 1 --eval_batch_size 32 --save_model
python -m Scripts.atlop.train_graph --model gat --epochs 15 --distant_limit 20000 --distant_epochs 1 --eval_batch_size 32 --save_model
```

산출된 `results/atlop_gcn.pt` / `results/atlop_gat.pt`를 로컬 `results/`로 복사한 뒤 아래 셀 실행. (빠른 대안: `--init_ckpt results/atlop.pt --distant_mode none --epochs 5`로 warm-start fine-tune — 이 경우 공정 비교를 위해 baseline도 같은 에폭 추가 학습한 대조군 필요, `Scripts/atlop/README.md` 참고.)

**개선 판정 기준**
1. **정량**: 학습 로그의 dev F1 / Ign F1이 baseline(61.71 / 59.86) 대비 상승했는가 (시드 변동 ±1점 감안 — `results/comparison.md`에 기록)
2. **정성(아래 셀)**: 명시 관계 (0,1)·(1,2)에 더해 **조합 관계 (0,2)** 를 검출하는가 — baseline은 셋 다 실패했다

In [3]:
# 개선 1(GCN)·개선 2(GAT) 체크포인트를 로드해 테스트 1을 baseline과 동일 조건으로 재실행.
# 그래프 하이퍼파라미터(--graph_layers/--graph_dim/--graph_heads)를 바꿔 학습했다면
# 아래 graph_kwargs 를 학습 시 값과 맞출 것 (기본값으로 학습했으면 그대로 두면 됨).
from Scripts.atlop.re_model_gcn import DocREModelGCN
from Scripts.atlop.re_model_gat import DocREModelGAT


def load_improved(cls, ckpt, **graph_kwargs):
    enc = AutoModel.from_pretrained(MODEL_NAME, config=config, attn_implementation="eager")
    m = cls(config, enc, num_labels=NUM_CLASSES, **graph_kwargs).to(device)
    state = torch.load(ckpt, map_location=device)
    missing, unexpected = m.load_state_dict(state, strict=False)
    assert not unexpected, f"체크포인트에 모르는 키가 있음(하이퍼파라미터 불일치?): {unexpected[:3]}"
    if missing:
        print(f"[경고] 그래프 파라미터 {len(missing)}개가 체크포인트에 없음 — 그래프 미학습(=baseline과 동일 출력)")
    return m.eval()


improved = [
    ("개선 1 — GCN (Entity Pair Graph + relational GCN)", DocREModelGCN, ROOT / "results" / "atlop_gcn.pt", {}),
    ("개선 2 — GAT (LCP + Entity Pair Graph + GAT)", DocREModelGAT, ROOT / "results" / "atlop_gat.pt", {}),
]
POI = [(0, 1), (1, 2), (0, 2)]  # 명시 (0,1)·(1,2) + 조합 목표 (0,2)

baseline_model = model  # probe()가 전역 model을 쓰므로 잠시 바꿔치기하고 마지막에 복원
try:
    print("=" * 70)
    print("[baseline] ATLOP — results/atlop.pt")
    probe(doc_multihop, pairs_of_interest=POI)
    for name, cls, ckpt, kwargs in improved:
        print("=" * 70)
        print(f"[{name}] — {ckpt.name}")
        if not ckpt.exists():
            print("체크포인트 없음 — Colab에서 train_graph.py로 학습 후 results/ 로 복사하세요.")
            continue
        model = load_improved(cls, ckpt, **kwargs)
        probe(doc_multihop, pairs_of_interest=POI)
finally:
    model = baseline_model  # 이후 셀들이 다시 baseline을 쓰도록 복원

[baseline] ATLOP — results/atlop.pt
(Tressonia) --P36:capital--> (Marniva)
[미검출] (Veldora Castle, Marniva) 쌍에서 관계를 찾지 못함
[미검출] (Marniva, Tressonia) 쌍에서 관계를 찾지 못함
[미검출] (Veldora Castle, Tressonia) 쌍에서 관계를 찾지 못함
[개선 1 — GCN (Entity Pair Graph + relational GCN)] — atlop_gcn.pt
(Tressonia) --P36:capital--> (Marniva)
[미검출] (Veldora Castle, Marniva) 쌍에서 관계를 찾지 못함
[미검출] (Marniva, Tressonia) 쌍에서 관계를 찾지 못함
[미검출] (Veldora Castle, Tressonia) 쌍에서 관계를 찾지 못함
[개선 2 — GAT (LCP + Entity Pair Graph + GAT)] — atlop_gat.pt
(Tressonia) --P36:capital--> (Marniva)
[미검출] (Veldora Castle, Marniva) 쌍에서 관계를 찾지 못함
[미검출] (Marniva, Tressonia) 쌍에서 관계를 찾지 못함
[미검출] (Veldora Castle, Tressonia) 쌍에서 관계를 찾지 못함


## 개선 3 — 통합 모델 (RoBERTa + DREEAM evidence + GREP GAT + PU 0.7)

세 개선을 하나로 합친 `DocREModelFull`(`Scripts/atlop/re_model_full.py`). 파이프라인: Document → **RoBERTa** → Entity Marker+logsumexp → **Evidence-guided Local Context(DREEAM)** → Entity Pair Rep → **Entity Pair Graph+GAT(GREP)** → Relation Classifier → **PU ATLoss(w=0.7)** → Triples.

위 개선 1·2와 달리 **인코더가 RoBERTa이고 evidence-guided context가 문장 span(`sent_pos`)을 필요로 하므로**, BERT 토크나이저·baseline 전처리를 쓰는 전역 `probe()`를 재사용할 수 없다 → 아래 셀에 RoBERTa 토크나이저 + `build_features_full` 기반의 전용 `probe_full()`을 따로 둔다.

- 학습: `python -m Scripts.atlop.train_full --epochs 15 --distant_limit 20000 --distant_epochs 1 --eval_batch_size 32 --save_model` → `results/atlop_full.pt`
- 그래프 하이퍼파라미터(`--graph_layers/--graph_dim/--graph_heads`)나 인코더(`--model_name_or_path`)를 바꿔 학습했다면 아래 `FULL_MODEL_NAME`·`probe_full(..., graph_dim=?, graph_heads=?)`를 학습 시 값과 맞출 것.
- 참고: 이 probe(허구 지명 multi-hop)는 개선 1·2에서 **판정 보류**였다(baseline이 전제 관계부터 미검출 → 그래프가 결합할 입력 없음). 통합 모델도 같은 한계에 걸릴 수 있으니, 정량 비교(dev F1/Ign F1, `results/comparison2.md`)를 주 근거로 볼 것.

In [4]:
# 개선 3 — 통합 모델(DocREModelFull) 로드 + 전용 probe로 테스트 1 재실행.
# RoBERTa 인코더 + sent_pos가 필요해 위 probe()를 못 쓰므로 전용 로더/프로브를 둔다.
# (train_full 은 import하지 않는다 — PUATLoss 등 학습 전용 의존성을 추론 경로에 끌어오지 않기 위해.)
from transformers import AutoConfig as _AutoConfig, AutoModel as _AutoModel, AutoTokenizer as _AutoTokenizer

from Scripts.atlop.re_model_full import DocREModelFull
from Scripts.atlop.preprocess_full import build_features_full

FULL_MODEL_NAME = "roberta-base"                 # 학습 시 --model_name_or_path 와 일치시킬 것
FULL_CKPT = ROOT / "results" / "atlop_full.pt"

_full_tok = _AutoTokenizer.from_pretrained(FULL_MODEL_NAME)
_full_cfg = _AutoConfig.from_pretrained(FULL_MODEL_NAME, num_labels=NUM_CLASSES)
_full_enc = _AutoModel.from_pretrained(FULL_MODEL_NAME, config=_full_cfg, attn_implementation="eager")
_full_cfg.cls_token_id = _full_tok.cls_token_id
_full_cfg.sep_token_id = _full_tok.sep_token_id


def probe_full(doc, pairs_of_interest=None, **model_kwargs):
    """통합 모델 전용 probe. RoBERTa 토크나이저 + build_features_full(sent_pos) 사용.
    출력 형식은 전역 probe()와 동일. graph_dim/graph_heads 등을 바꿔 학습했다면
    model_kwargs 로 넘겨 구조를 맞출 것 (기본값 학습이면 비워두면 됨)."""
    m = DocREModelFull(_full_cfg, _full_enc, num_labels=NUM_CLASSES, **model_kwargs).to(device)
    state = torch.load(FULL_CKPT, map_location=device)
    missing, unexpected = m.load_state_dict(state, strict=False)
    assert not unexpected, f"체크포인트에 모르는 키(하이퍼파라미터/인코더 불일치?): {unexpected[:3]}"
    if missing:
        print(f"[경고] 파라미터 {len(missing)}개 누락 — 구조/하이퍼파라미터 확인 필요")
    m.eval()

    # 단일 문서라 collate 없이 배치를 직접 구성 (모두 1-문서 리스트로 감쌈)
    feats = build_features_full([doc], _full_tok, rel2id, show_progress=False)
    f = feats[0]
    input_ids = torch.tensor([f["input_ids"]], dtype=torch.long, device=device)
    attention_mask = torch.ones_like(input_ids)
    with torch.no_grad():
        preds = m(
            input_ids=input_ids,
            attention_mask=attention_mask,
            entity_pos=[f["entity_pos"]],
            hts=[f["hts"]],
            sent_pos=[f["sent_pos"]],   # evidence-guided context 에 필요
        )[0].cpu()

    names = [v[0]["name"] for v in doc["vertexSet"]]
    found = []
    for (h, t), row in zip(f["hts"], preds):
        for r in range(1, NUM_CLASSES):
            if row[r] == 1:
                found.append((h, t, id2rel[r]))
    if not found:
        print("예측된 관계 없음 (모두 Na)")
    for h, t, p in found:
        mark = "  <-- 관심 쌍" if pairs_of_interest and (h, t) in pairs_of_interest else ""
        print(f"({names[h]}) --{p}:{rel_names.get(p, '?')}--> ({names[t]}){mark}")
    if pairs_of_interest:
        for h, t in pairs_of_interest:
            if not any(fh == h and ft == t for fh, ft, _ in found):
                print(f"[미검출] ({names[h]}, {names[t]}) 쌍에서 관계를 찾지 못함")
    return found


print("=" * 70)
print(f"[개선 3 — 통합 모델 (RoBERTa + DREEAM evidence + GREP GAT + PU 0.7)] — {FULL_CKPT.name}")
if not FULL_CKPT.exists():
    print("체크포인트 없음 — Colab에서 train_full.py로 학습 후 results/ 로 복사하세요.")
else:
    probe_full(doc_multihop, pairs_of_interest=POI);   # POI = [(0,1),(1,2),(0,2)] (위 셀에서 정의)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[개선 3 — 통합 모델 (RoBERTa + DREEAM evidence + GREP GAT + PU 0.7)] — atlop_full.pt
(Tressonia) --P36:capital--> (Marniva)
[미검출] (Veldora Castle, Marniva) 쌍에서 관계를 찾지 못함
[미검출] (Marniva, Tressonia) 쌍에서 관계를 찾지 못함
[미검출] (Veldora Castle, Tressonia) 쌍에서 관계를 찾지 못함
